# Crypto price cleaning with yfinance

This notebook downloads daily market data for:
- VIX (`^VIX`)
- Ethereum (`ETH-USD`)
- Bitcoin (`BTC-USD`)

Then it cleans the relevant columns and exports a ready-to-use CSV file.

In [63]:
import numpy as np
import pandas as pd
import yfinance as yf

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

In [64]:
# Configuration
TICKERS = {
    "VIX": "^VIX",
    "ETH": "ETH-USD",
    "BTC": "BTC-USD",
    "GC=F": "GC=F",
}

START_DATE = "2018-01-01"
END_DATE = None  # None means up to today

In [65]:
# Download price history from Yahoo Finance
df_raw = yf.download(
    tickers=list(TICKERS.values()),
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=False,
    group_by="ticker",
    progress=False,
)

if df_raw.empty:
    raise ValueError("No data was downloaded. Check internet connection or ticker symbols.")

print("Raw dataframe shape:", df_raw.shape)
df_raw.head()

Raw dataframe shape: (3028, 24)


Ticker             GC=F                                                              ^VIX                                         ETH-USD  \
Price              Open         High          Low        Close    Adj Close Volume   Open   High   Low Close Adj Close Volume        Open   
Date                                                                                                                                        
2018-01-01          NaN          NaN          NaN          NaN          NaN    NaN    NaN    NaN   NaN   NaN       NaN    NaN  755.757019   
2018-01-02  1302.300049  1317.599976  1302.300049  1313.699951  1313.699951   68.0  10.95  11.07  9.52  9.77      9.77    0.0  772.346008   
2018-01-03  1320.000000  1320.099976  1312.099976  1316.199951  1316.199951   42.0   9.56   9.65  8.94  9.15      9.15    0.0  886.000000   
2018-01-04  1319.400024  1322.000000  1319.400024  1319.400024  1319.400024    2.0   9.01   9.31  8.92  9.22      9.22    0.0  961.713013   
2018-01-05  1320.300049  1320.300049  1320.300049  1320.300049  1320.300049    1.0   9.10   9.54  9.00  9.22      9.22    0.0  975.750000   

Ticker                                                                        BTC-USD                                            \
Price              High         Low       Close   Adj Close      Volume          Open          High           Low         Close   
Date                                                                                                                              
2018-01-01   782.530029  742.004028  772.640991  772.640991  2595760128  14112.200195  14112.200195  13154.700195  13657.200195   
2018-01-02   914.830017  772.346008  884.443970  884.443970  5783349760  13625.000000  15444.599609  13163.599609  14982.099609   
2018-01-03   974.471008  868.450989  962.719971  962.719971  5093159936  14978.200195  15572.799805  14844.500000  15201.000000   
2018-01-04  1045.079956  946.085999  980.921997  980.921997  6502859776  15270.700195  15739.700195  14522.200195  15599.200195   
2018-01-05  1075.390015  956.325012  997.719971  997.719971  6683149824  15477.200195  17705.199219  15202.799805  17429.500000   

Ticker                                 
Price          Adj Close       Volume  
Date                                   
2018-01-01  13657.200195  10291200000  
2018-01-02  14982.099609  16846600192  
2018-01-03  15201.000000  16871900160  
2018-01-04  15599.200195  21783199744  
2018-01-05  17429.500000  23840899072

In [66]:
# Keep adjusted close and flatten names
adj_close = df_raw.xs("Adj Close", axis=1, level=1)

symbol_to_name = {symbol: name for name, symbol in TICKERS.items()}
df_prices = adj_close.rename(columns=symbol_to_name)

# Fallback: if a ticker is missing from the multi-download result, fetch it separately
missing_symbols = [s for s in TICKERS.values() if s not in adj_close.columns]
for symbol in missing_symbols:
    one = yf.download(
        tickers=symbol,
        start=START_DATE,
        end=END_DATE,
        interval="1d",
        auto_adjust=False,
        progress=False,
    )
    if not one.empty and "Adj Close" in one.columns:
        col_name = symbol_to_name.get(symbol, symbol)
        df_prices[col_name] = pd.to_numeric(one["Adj Close"], errors="coerce")
        print(f"Fetched missing symbol separately: {symbol}")

# Ensure clean datetime index and sorted rows
df_prices.index = pd.to_datetime(df_prices.index, errors="coerce")
df_prices = df_prices[~df_prices.index.isna()]
df_prices = df_prices.sort_index()
df_prices = df_prices[~df_prices.index.duplicated(keep="last")]

# Ensure numeric dtype for modeling
df_prices = df_prices.apply(pd.to_numeric, errors="coerce")

print("Clean price dataframe shape:", df_prices.shape)
print("Columns:", df_prices.columns.tolist())
df_prices.head()

Clean price dataframe shape: (3028, 4)
Columns: ['GC=F', 'VIX', 'ETH', 'BTC']


Ticker,GC=F,VIX,ETH,BTC
Date,,,,
2018-01-01,NaN,NaN,772.640991,13657.200195
2018-01-02,1313.699951,9.77,884.443970,14982.099609
2018-01-03,1316.199951,9.15,962.719971,15201.000000
2018-01-04,1319.400024,9.22,980.921997,15599.200195
2018-01-05,1320.300049,9.22,997.719971,17429.500000


In [67]:
# Data quality checks
print("Date range:", df_prices.index.min().date(), "to", df_prices.index.max().date())
print("Columns:", df_prices.columns.tolist())
print("Missing values per column:")
print(df_prices.isna().sum())
print("Duplicate dates:", int(df_prices.index.duplicated().sum()))

Date range: 2018-01-01 to 2026-04-16
Columns: ['GC=F', 'VIX', 'ETH', 'BTC']
Missing values per column:
Ticker
GC=F    944
VIX     945
ETH       0
BTC       0
dtype: int64
Duplicate dates: 0


In [68]:
# Keep only dates where core assets are available
required_cols = [col for col in ["VIX", "ETH", "BTC"] if col in df_prices.columns]
df_final = df_prices.dropna(subset=required_cols).copy()

# Build volume table for OBV
vol_raw = df_raw.xs("Volume", axis=1, level=1)
vol_cols = {symbol: name for name, symbol in TICKERS.items() if symbol in vol_raw.columns}
df_volume = vol_raw.rename(columns=vol_cols)
df_volume.index = pd.to_datetime(df_volume.index, errors="coerce")
df_volume = df_volume.sort_index()


def add_technical_indicators(df, volume_df, price_col):
    """Add RSI, MACD, EMA/SMA, Bollinger, volatility and OBV for one price column."""
    out = df.copy()
    p = out[price_col]
    prefix = price_col.lower()

    # Returns / momentum
    out[f"{prefix}_ret_1d"] = p.pct_change(1)
    out[f"{prefix}_ret_3d"] = p.pct_change(3)
    out[f"{prefix}_ret_7d"] = p.pct_change(7)

    # EMA/SMA
    out[f"{prefix}_sma_10"] = p.rolling(10).mean()
    out[f"{prefix}_sma_30"] = p.rolling(30).mean()
    out[f"{prefix}_ema_12"] = p.ewm(span=12, adjust=False).mean()
    out[f"{prefix}_ema_26"] = p.ewm(span=26, adjust=False).mean()
    out[f"{prefix}_sma_ratio_10"] = p / out[f"{prefix}_sma_10"] - 1
    out[f"{prefix}_ema_ratio_12_26"] = out[f"{prefix}_ema_12"] / out[f"{prefix}_ema_26"] - 1

    # Volatility + Bollinger z-score
    out[f"{prefix}_vol_7d"] = out[f"{prefix}_ret_1d"].rolling(7).std()
    out[f"{prefix}_vol_14d"] = out[f"{prefix}_ret_1d"].rolling(14).std()
    bb_mid = p.rolling(20).mean()
    bb_std = p.rolling(20).std()
    out[f"{prefix}_bb_z_20"] = (p - bb_mid) / (bb_std + 1e-9)

    # RSI(14)
    delta = p.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / (loss + 1e-9)
    out[f"{prefix}_rsi_14"] = 100 - (100 / (1 + rs))

    # MACD(12,26,9)
    macd = out[f"{prefix}_ema_12"] - out[f"{prefix}_ema_26"]
    macd_signal = macd.ewm(span=9, adjust=False).mean()
    out[f"{prefix}_macd"] = macd
    out[f"{prefix}_macd_signal"] = macd_signal
    out[f"{prefix}_macd_hist"] = macd - macd_signal

    # OBV (if volume exists)
    if price_col in volume_df.columns:
        vol = pd.to_numeric(volume_df[price_col], errors="coerce").reindex(out.index)
        direction = np.sign(p.diff()).fillna(0.0)
        out[f"{prefix}_obv"] = (direction * vol.fillna(0)).cumsum()
    else:
        out[f"{prefix}_obv"] = np.nan

    return out


# Build technical-indicator feature set
df_features = df_final.copy()
for col in ["ETH", "BTC", "VIX", "GC=F", "GOLD"]:
    if col in df_features.columns:
        df_features = add_technical_indicators(df_features, df_volume, col)

# Cross-asset feature
if {"BTC", "ETH"}.issubset(df_features.columns):
    df_features["btc_eth_spread_1d"] = df_features["btc_ret_1d"] - df_features["eth_ret_1d"]

print("Final clean dataframe shape:", df_final.shape)
print("Feature dataframe shape:", df_features.shape)
df_features.tail()

Final clean dataframe shape: (2083, 4)
Feature dataframe shape: (2083, 73)


/var/folders/n5/f_gk0vdx3wn0mvvk511jlsj00000gn/T/ipykernel_33625/2242791988.py:20: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out[f"{prefix}_ret_1d"] = p.pct_change(1)
/var/folders/n5/f_gk0vdx3wn0mvvk511jlsj00000gn/T/ipykernel_33625/2242791988.py:21: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out[f"{prefix}_ret_3d"] = p.pct_change(3)
/var/folders/n5/f_gk0vdx3wn0mvvk511jlsj00000gn/T/ipykernel_33625/2242791988.py:22: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to callin

Ticker,GC=F,VIX,ETH,BTC,eth_ret_1d,eth_ret_3d,eth_ret_7d,eth_sma_10,eth_sma_30,eth_ema_12,eth_ema_26,eth_sma_ratio_10,eth_ema_ratio_12_26,eth_vol_7d,eth_vol_14d,eth_bb_z_20,eth_rsi_14,eth_macd,eth_macd_signal,eth_macd_hist,eth_obv,btc_ret_1d,btc_ret_3d,btc_ret_7d,btc_sma_10,btc_sma_30,btc_ema_12,btc_ema_26,btc_sma_ratio_10,btc_ema_ratio_12_26,btc_vol_7d,btc_vol_14d,btc_bb_z_20,btc_rsi_14,btc_macd,btc_macd_signal,btc_macd_hist,btc_obv,vix_ret_1d,vix_ret_3d,vix_ret_7d,vix_sma_10,vix_sma_30,vix_ema_12,vix_ema_26,vix_sma_ratio_10,vix_ema_ratio_12_26,vix_vol_7d,vix_vol_14d,vix_bb_z_20,vix_rsi_14,vix_macd,vix_macd_signal,vix_macd_hist,vix_obv,gc=f_ret_1d,gc=f_ret_3d,gc=f_ret_7d,gc=f_sma_10,gc=f_sma_30,gc=f_ema_12,gc=f_ema_26,gc=f_sma_ratio_10,gc=f_ema_ratio_12_26,gc=f_vol_7d,gc=f_vol_14d,gc=f_bb_z_20,gc=f_rsi_14,gc=f_macd,gc=f_macd_signal,gc=f_macd_hist,gc=f_obv,btc_eth_spread_1d
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-04-10,4761.899902,19.230000,2245.148682,72979.046875,0.025583,0.001491,0.066727,2128.928101,2111.551554,2156.771850,2148.520176,0.054591,0.003841,0.033911,0.031493,1.003398,56.838982,8.251674,-12.365318,20.616991,7.769937e+11,0.016877,0.014433,0.069552,69290.102344,69984.505729,70344.593228,70410.543090,0.053239,-0.000937,0.022346,0.022947,1.150563,57.011389,-65.949862,-650.362103,584.412241,7.017386e+11,-0.013340,-0.254073,-0.238416,24.503,24.703667,23.403957,23.793992,-0.215198,-0.016392,0.078335,0.087868,-1.875942,34.535025,-0.390036,0.702226,-1.092262,0.0,-0.006323,0.022503,0.024593,4671.780029,4837.443327,4701.355881,4755.044520,0.019290,-0.011291,0.018409,0.023911,0.419647,57.937490,-53.688638,-84.610580,30.921942,1520155.0,-0.008706
2026-04-13,4742.399902,19.120001,2370.714844,74484.640625,0.055928,0.082352,0.108465,2166.872388,2126.216654,2189.686157,2164.979040,0.094072,0.011412,0.038219,0.034441,1.993158,62.992076,24.707116,-4.950831,29.657947,8.007029e+11,0.020630,0.047260,0.094098,70104.728906,70271.267187,70981.523597,70712.328093,0.062477,0.003807,0.021947,0.023440,1.597686,59.580235,269.195504,-466.450582,735.646086,7.540168e+11,-0.005720,-0.091255,-0.220864,23.310,24.679000,22.744887,23.447771,-0.179751,-0.029977,0.079131,0.087936,-1.662176,35.286733,-0.702884,0.421204,-1.124088,0.0,-0.004095,-0.001495,-0.008530,4696.820020,4821.173324,4707.670346,4754.107881,0.009704,-0.009768,0.014616,0.021203,0.442677,65.965054,-46.437535,-76.975971,30.538436,1520123.0,-0.035297
2026-04-14,4825.000000,18.360001,2323.308594,74181.609375,-0.019997,0.061286,0.129546,2196.851758,2136.084644,2210.243455,2176.707155,0.057563,0.015407,0.034365,0.035200,1.580090,59.469059,33.536299,2.746595,30.789704,7.755177e+11,-0.004068,0.033633,0.109033,70853.745313,70451.459115,71473.844486,70969.311891,0.046968,0.007109,0.019248,0.023397,1.526206,59.882263,504.532594,-272.253947,776.786541,7.004759e+11,-0.039749,-0.057978,-0.230834,22.085,24.576334,22.070289,23.070899,-0.168667,-0.043371,0.079145,0.086928,-1.673176,31.991613,-1.000610,0.136841,-1.137451,0.0,0.017417,0.006844,0.037300,4726.720020,4805.526660,4725.721062,4759.359149,0.020792,-0.007068,0.010328,0.021336,1.017489,68.715374,-33.638087,-68.308395,34.670307,1520411.0,0.015928
2026-04-15,4800.000000,18.170000,2359.437012,74805.078125,0.015550,0.050905,0.119404,2222.324634,2148.640247,2233.196310,2190.242700,0.061698,0.019611,0.034242,0.035293,1.853893,60.530977,42.953609,10.787998,32.165611,7.925221e+11,0.008405,0.025021,0.086338,71510.921875,70668.506771,71986.341969,71253.442724,0.046065,0.010286,0.018238,0.023341,1.711431,59.514129,732.899245,-71.223308,804.122553,7.385661e+11,-0.010349,-0.055122,-0.248242,21.377,24.396334,21.470244,22.707869,-0.150021,-0.054502,0.077357,0.086326,-1.563681,34.032113,-1.237625,-0.138052,-1.099573,0.0,-0.005181,0.008001,0.030751,4741.960010,4795.279997,4737.148591,4762.369583,0.012240,-0.005296,0.011005,0.020003,1.030626,62.364104,-25.220992,-59.690914,34.469922,1520123.0,-0.007146
2026-04-16,4803.6

In [69]:
# Export base prices and indicator-rich feature file
base_output_path = "crypto_prices_clean.csv"
feature_output_path = "crypto_prices_with_indicators.csv"

df_final.to_csv(base_output_path, index=True)
df_features.to_csv(feature_output_path, index=True)

print(f"Saved: {base_output_path}")
print(f"Saved: {feature_output_path}")

Saved: crypto_prices_clean.csv
Saved: crypto_prices_with_indicators.csv


## Macro + Sentiment (Alpha Vantage)

This section fetches:
- ETH market news sentiment
- Federal Funds Rate (daily)
- CPI (monthly, forward-filled to daily)
- 10Y Treasury Yield (daily)

Then it transforms all series to a daily index and merges them with the cleaned crypto prices.

In [70]:
import time
import requests

# Use your Alpha Vantage key directly here (recommended: move to .env later)
ALPHA_VANTAGE_API_KEY = "T90NCBSD09SEGURM"

START_AV = "20180101T0000"
END_AV = pd.Timestamp.today().strftime("%Y%m%dT0000")
START_DATE_DAILY = pd.Timestamp("2018-01-01")
END_DATE_DAILY = pd.Timestamp.today().normalize()


def fetch_av_json(url, max_retries=6, base_wait=2.0):
    """Fetch Alpha Vantage JSON with retry/backoff for free-tier limits."""
    for attempt in range(1, max_retries + 1):
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        payload = r.json()

        # Alpha Vantage rate limit response
        if isinstance(payload, dict) and "Information" in payload:
            wait = base_wait * attempt
            print(f"Rate-limit/info response (attempt {attempt}/{max_retries}). Sleeping {wait:.1f}s...")
            time.sleep(wait)
            continue

        return payload

    raise RuntimeError("Alpha Vantage request failed after retries due to limits/info responses.")


# --- 1) ETH NEWS SENTIMENT ---
news_url = (
    "https://www.alphavantage.co/query"
    f"?function=NEWS_SENTIMENT&tickers=ETH&apikey={ALPHA_VANTAGE_API_KEY}"
    f"&time_from={START_AV}&time_to={END_AV}&limit=1000"
)
news_payload = fetch_av_json(news_url)
feed = news_payload.get("feed", [])

news_rows = []
for item in feed:
    published = item.get("time_published")
    if not published:
        continue

    # Prefer ETH-specific sentiment if available, fallback to overall sentiment
    ticker_items = item.get("ticker_sentiment", []) or []
    eth_items = [t for t in ticker_items if str(t.get("ticker", "")).upper() == "ETH"]

    if eth_items:
        t0 = eth_items[0]
        sent = float(t0.get("ticker_sentiment_score", 0.0))
        rel = float(t0.get("relevance_score", 0.0))
    else:
        sent = float(item.get("overall_sentiment_score", 0.0))
        rel = 1.0

    dt = pd.to_datetime(published, format="%Y%m%dT%H%M%S", errors="coerce")
    if pd.isna(dt):
        continue

    news_rows.append({
        "date": dt.normalize(),
        "sent_score": sent,
        "sent_weight": rel,
    })

if news_rows:
    news_df = pd.DataFrame(news_rows)
    news_daily = (
        news_df.groupby("date", as_index=True)
        .apply(
            lambda g: pd.Series(
                {
                    "eth_news_sent_mean": g["sent_score"].mean(),
                    "eth_news_sent_wmean": np.average(g["sent_score"], weights=np.maximum(g["sent_weight"], 1e-9)),
                    "eth_news_sent_std": g["sent_score"].std(ddof=0),
                    "eth_news_count": len(g),
                    "eth_news_pos_ratio": (g["sent_score"] > 0.15).mean(),
                    "eth_news_neg_ratio": (g["sent_score"] < -0.15).mean(),
                }
            )
        )
        .reset_index()
        .set_index("date")
    )
else:
    news_daily = pd.DataFrame(
        columns=[
            "eth_news_sent_mean",
            "eth_news_sent_wmean",
            "eth_news_sent_std",
            "eth_news_count",
            "eth_news_pos_ratio",
            "eth_news_neg_ratio",
        ]
    )


# --- 2) FED FUNDS RATE (daily) ---
fed_url = (
    "https://www.alphavantage.co/query"
    f"?function=FEDERAL_FUNDS_RATE&interval=daily&apikey={ALPHA_VANTAGE_API_KEY}"
)
fed_payload = fetch_av_json(fed_url)
fed_data = pd.DataFrame(fed_payload.get("data", []))
if not fed_data.empty:
    fed_data["date"] = pd.to_datetime(fed_data["date"], errors="coerce")
    fed_data["federal_funds_rate"] = pd.to_numeric(fed_data["value"], errors="coerce")
    fed_daily = fed_data[["date", "federal_funds_rate"]].dropna().set_index("date").sort_index()
else:
    fed_daily = pd.DataFrame(columns=["federal_funds_rate"])


# --- 3) CPI (monthly -> daily via forward fill) ---
cpi_url = (
    "https://www.alphavantage.co/query"
    f"?function=CPI&interval=monthly&apikey={ALPHA_VANTAGE_API_KEY}"
)
cpi_payload = fetch_av_json(cpi_url)
cpi_data = pd.DataFrame(cpi_payload.get("data", []))
if not cpi_data.empty:
    cpi_data["date"] = pd.to_datetime(cpi_data["date"], errors="coerce")
    cpi_data["cpi"] = pd.to_numeric(cpi_data["value"], errors="coerce")
    cpi_monthly = cpi_data[["date", "cpi"]].dropna().set_index("date").sort_index()
else:
    cpi_monthly = pd.DataFrame(columns=["cpi"])


# --- 4) 10Y TREASURY YIELD (daily) ---
ty_url = (
    "https://www.alphavantage.co/query"
    f"?function=TREASURY_YIELD&interval=daily&maturity=10year&apikey={ALPHA_VANTAGE_API_KEY}"
)
ty_payload = fetch_av_json(ty_url)
ty_data = pd.DataFrame(ty_payload.get("data", []))
if not ty_data.empty:
    ty_data["date"] = pd.to_datetime(ty_data["date"], errors="coerce")
    ty_data["treasury_10y"] = pd.to_numeric(ty_data["value"], errors="coerce")
    ty_daily = ty_data[["date", "treasury_10y"]].dropna().set_index("date").sort_index()
else:
    ty_daily = pd.DataFrame(columns=["treasury_10y"])

print("News daily shape:", news_daily.shape)
print("Fed daily shape:", fed_daily.shape)
print("CPI monthly shape:", cpi_monthly.shape)
print("Treasury daily shape:", ty_daily.shape)

/var/folders/n5/f_gk0vdx3wn0mvvk511jlsj00000gn/T/ipykernel_33625/2459165650.py:73: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Rate-limit/info response (attempt 1/6). Sleeping 2.0s...
Rate-limit/info response (attempt 1/6). Sleeping 2.0s...
Rate-limit/info response (attempt 1/6). Sleeping 2.0s...
News daily shape: (121, 6)
Fed daily shape: (26221, 1)
CPI monthly shape: (1358, 1)
Treasury daily shape: (16055, 1)


In [ ]:
full_daily_index = pd.date_range(START_DATE_DAILY, END_DATE_DAILY, freq="D")

news_daily_full = news_daily.reindex(full_daily_index)
if not news_daily_full.empty:
    # No news day: 0 count and neutral sentiment values
    fill_zero_cols = [c for c in news_daily_full.columns if c in [
        "eth_news_sent_mean", "eth_news_sent_wmean", "eth_news_sent_std",
        "eth_news_count", "eth_news_pos_ratio", "eth_news_neg_ratio"
    ]]
    news_daily_full[fill_zero_cols] = news_daily_full[fill_zero_cols].fillna(0)

fed_daily_full = fed_daily.reindex(full_daily_index).ffill()
cpi_daily_full = cpi_monthly.reindex(full_daily_index).ffill()
ty_daily_full = ty_daily.reindex(full_daily_index).ffill()

macro_sentiment_daily = pd.concat(
    [news_daily_full, fed_daily_full, cpi_daily_full, ty_daily_full],
    axis=1,
)
macro_sentiment_daily.index.name = "Date"

macro_sentiment_daily = macro_sentiment_daily.loc[
    (macro_sentiment_daily.index >= START_DATE_DAILY) & (macro_sentiment_daily.index <= END_DATE_DAILY)
]

# Merge with cleaned crypto prices (date index)
crypto_macro_merged = df_final.join(macro_sentiment_daily, how="left")

# Add simple macro deltas
if "federal_funds_rate" in crypto_macro_merged.columns:
    crypto_macro_merged["fed_rate_change_7d"] = crypto_macro_merged["federal_funds_rate"].diff(7)
if "treasury_10y" in crypto_macro_merged.columns:
    crypto_macro_merged["treasury_10y_change_7d"] = crypto_macro_merged["treasury_10y"].diff(7)
if "cpi" in crypto_macro_merged.columns:
    crypto_macro_merged["cpi_yoy_pct"] = crypto_macro_merged["cpi"].pct_change(365)

# Export
macro_output_path = "macro_sentiment_daily.csv"
merged_output_path = "crypto_prices_macro_sentiment.csv"

macro_sentiment_daily.to_csv(macro_output_path, index=True)
crypto_macro_merged.to_csv(merged_output_path, index=True)

print(f"Saved macro/sentiment daily file: {macro_output_path}")
print(f"Saved merged crypto+macro+sentiment file: {merged_output_path}")
print("Merged shape:", crypto_macro_merged.shape)
crypto_macro_merged.tail()

Saved macro/sentiment daily file: macro_sentiment_daily.csv
Saved merged crypto+macro+sentiment file: crypto_prices_macro_sentiment.csv
Merged shape: (2083, 16)


,GC=F,VIX,ETH,BTC,eth_news_sent_mean,eth_news_sent_wmean,eth_news_sent_std,eth_news_count,eth_news_pos_ratio,eth_news_neg_ratio,federal_funds_rate,cpi,treasury_10y,fed_rate_change_7d,treasury_10y_change_7d,cpi_yoy_pct
Date,,,,,,,,,,,,,,,,
2026-04-10,4761.899902,19.230000,2245.148682,72979.046875,0.0,0.0,0.0,0.0,0.0,0.0,3.64,330.213,4.31,0.0,0.01,0.04609
2026-04-13,4742.399902,19.120001,2370.714844,74484.640625,0.0,0.0,0.0,0.0,0.0,0.0,3.64,330.213,4.30,0.0,-0.03,0.04609
2026-04-14,4825.000000,18.360001,2323.308594,74181.609375,0.0,0.0,0.0,0.0,0.0,0.0,3.64,330.213,4.26,0.0,-0.05,0.04609
2026-04-15,4800.000000,18.170000,2359.437012,74805.078125,0.0,0.0,0.0,0.0,0.0,0.0,3.64,330.213,4.26,0.0,-0.08,0.04609
2026-04-16,4803.600098,18.500000,2327.979980,74431.078125,0.0,0.0,0.0,0.0,0.0,0.0,3.64,330.213,4.26,0.0,-0.07,0.04609


## Notes

Technical indicators are now calculated in the main feature-engineering cell (Cell 6):
- RSI(14)
- MACD(12,26,9)
- EMA/SMA features
- OBV (when volume is available)

The Alpha Vantage sentiment/macro section starts below.